# 2.4 Circuit Mechanics

This notebook covers registers, circuit properties, `measure_all`, and transpilation into a basis-gate set.

In [ ]:
import numpy as np
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
from qiskit.quantum_info import Statevector

In [ ]:
data = QuantumRegister(2, "data")
ancilla = QuantumRegister(1, "anc")
readout = ClassicalRegister(3, "c")

ideal = QuantumCircuit(data, ancilla, name="ideal_demo")
ideal.h(data[0])
ideal.cx(data[0], data[1])
ideal.ry(np.pi / 3, ancilla[0])
ideal.cz(data[1], ancilla[0])
ideal.barrier()

measured = QuantumCircuit(data, ancilla, readout, name="measured_demo")
measured.compose(ideal, inplace=True)
measured.measure(data[0], readout[0])
measured.measure(data[1], readout[1])
measured.measure(ancilla[0], readout[2])

measured

In [ ]:
state = Statevector.from_instruction(ideal)
print("num_qubits:", measured.num_qubits)
print("num_clbits:", measured.num_clbits)
print("width:", measured.width())
print("size:", measured.size())
print("depth:", measured.depth())
print("count_ops:", measured.count_ops())
print("Ideal probabilities before measurement:", state.probabilities_dict(decimals=4))

## `measure_all`

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
measured_bell = bell.measure_all(inplace=False)
measured_bell

In [ ]:
print("Original clbits:", bell.num_clbits)
print("Measured clbits:", measured_bell.num_clbits)

## Transpilation

In [ ]:
transpiled = transpile(
    measured,
    basis_gates=["rz", "sx", "x", "cx", "measure"],
    optimization_level=1,
)
transpiled

In [ ]:
print("Original depth:", measured.depth())
print("Transpiled depth:", transpiled.depth())
print("Transpiled count_ops:", transpiled.count_ops())

## Practice

1. Add another `ClassicalRegister` and remap one of the measurements.
2. Insert or remove barriers and compare circuit depth.
3. Change the basis-gate list in `transpile()` and inspect the new decomposition.